# E2E 본 학습 on Colab — ResNet18×2 + ControlHead + WaypointHead (PHASE2 4단계)

`labels_cache.h5`(Phase 2 자동 라벨)로 `model.E2ENet`을 학습해 `e2e_best.pt` → `e2e.onnx`까지 만든다.

전제:
- Phase 1 SegFormer/YOLO **freeze 완료**, `extract_labels.py`로 `labels_cache.h5` 생성 + `visualize_labels.py` **눈 검증 통과**.
- 이 노트북은 H5만 있으면 된다 (SegFormer/YOLO/rosbag 불필요 — 합성은 데이터로더가 raw lane/front + seg/det에서 수행).

합성·정규화 계약은 `training/dataset.py`가 단일 소스다 (lane: seg 3채널 alpha-blend, front: car bbox 사각형, **BGR→RGB→ImageNet 정규화**). 추론 노드(rover_lane)도 이 변환과 픽셀 단위로 동일해야 한다.

흐름: 레포 clone → H5 업로드/Drive 마운트 → 학습(early-stop, best 저장) → ONNX export → 다운로드 → **Jetson에서만** `trtexec --fp16`로 engine 빌드.

## 1. 설치 + 레포

`dataset.py` / `model.py` / `train_e2e.py` / `export_onnx.py`를 그대로 쓰기 위해 레포를 clone한다. (이미 받았으면 이 셀 건너뛰고 `%cd`만.)

In [ ]:
!pip install -q h5py onnx onnxruntime
# 레포 clone (이미 있으면 에러 무시). 본인 fork/브랜치로 바꿔도 됨.
REPO = 'https://github.com/lsik209-arch/AUE4040-team-project.git'
import os
if not os.path.isdir('/content/AUE4040-team-project'):
    !git clone -q $REPO /content/AUE4040-team-project
%cd /content/AUE4040-team-project/final_project/training

import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda', torch.cuda.is_available())

## 2. labels_cache.h5 올리기

둘 중 하나. bag이 여러 개면 H5도 여러 개 — `CACHES` 리스트에 다 넣으면 합쳐서 학습한다.

In [ ]:
# (A) 브라우저 업로드
from google.colab import files
up = files.upload()                      # labels_cache.h5 선택 (여러 개 가능)
CACHES = [f'/content/{name}' for name in up]

# (B) Drive 마운트 (위 A 대신 주석 해제해서 사용)
# from google.colab import drive
# drive.mount('/content/drive')
# CACHES = ['/content/drive/MyDrive/aue4040/labels_cache.h5']

import h5py
for c in CACHES:
    with h5py.File(c, 'r') as f:
        print(c, '→ samples:', f['lane'].shape[0], '| keys:', list(f.keys()))

## 3. 합성 입력 눈 확인 (sanity)

데이터로더가 H5에서 만드는 **실제 학습 입력**(정규화 역변환)을 한 장 그려본다. seg 오버레이/bbox가 제대로 합성되는지, 추론 노드가 맞춰야 할 픽셀이 이거다. 여기서 이상하면 학습 무의미 → Phase 1로 되돌아감.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from dataset import E2EDataset, IMAGENET_MEAN, IMAGENET_STD

ds = E2EDataset(CACHES, augment=False)
print('total samples:', len(ds))

def denorm(t):
    # (3,H,W) 정규화 텐서 → RGB uint8 (시각화용 역변환)
    x = t.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(x * 255, 0, 255).astype(np.uint8)

idx = 0
lane_t, front_t, steer, thr, wp = ds[idx]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(denorm(lane_t));  ax[0].set_title('lane input (seg overlay)'); ax[0].axis('off')
ax[1].imshow(denorm(front_t)); ax[1].set_title('front input (bbox overlay)'); ax[1].axis('off')
plt.suptitle(f'idx={idx}  steer={steer:.2f}  throttle={thr:.2f}  wp[-1]={wp[-1].tolist()}')
plt.tight_layout(); plt.show()

## 4. 학습

`train_e2e.py`를 그대로 호출 (split·AdamW·cosine LR·early-stop·best 저장). 인자는 셀 아래에서 조절. best는 `/content/e2e_best.pt`.

- **의도 시각화**: `--viz_dir`을 켜면 매 `--viz_every` epoch + 새 best마다 **예측 vs GT**(waypoint 흰=GT/노랑=예측, steer/throttle 텍스트) 패널을 저장한다. 끄려면 그 두 인자만 빼면 됨.
- **이어학습**: 새 bag 추가 후 처음부터 안 돌리려면 `--resume /content/e2e_best.pt` (가중치+optimizer state 복원).

In [ ]:
import shlex, subprocess, sys

CKPT = '/content/e2e_best.pt'
VIZ_DIR = '/content/train_viz'          # 의도 시각화 끄려면 아래 --viz_dir 줄 제거
RESUME = ''                             # 이어학습하려면 RESUME = CKPT 로 설정

cache_args = ' '.join(f'--cache {shlex.quote(c)}' for c in CACHES)
resume_arg = f'--resume {shlex.quote(RESUME)}' if RESUME else ''
cmd = (f'{sys.executable} train_e2e.py {cache_args} '
       f'--out {CKPT} '
       f'--viz_dir {VIZ_DIR} --viz_every 5 '
       f'{resume_arg} '
       f'--epochs 60 --batch 32 --lr 3e-4 --val_frac 0.15 '
       f'--workers 2 --patience 10 --device cuda')
print(cmd)
# 실시간 로그가 보이게 직접 실행
!{cmd}

### 4-1. 의도 시각화 결과 보기

마지막 epoch에 저장된 예측 vs GT 패널들. 예측 waypoint(노랑)가 GT(흰)를 잘 따라가고 steer/throttle 예측이 GT에 가까우면 학습이 잘 된 것. (BGR 저장이라 matplotlib 표시 시 RGB로 뒤집어 본다.)

In [ ]:
import glob, os, cv2, matplotlib.pyplot as plt
pngs = sorted(glob.glob(f'{VIZ_DIR}/*.png'))
# 파일명 ep###_s##.png → 가장 큰 ep### 의 패널들만
last = []
if pngs:
    last_ep = max(os.path.basename(p).split('_')[0] for p in pngs)  # 'ep###'
    last = [p for p in pngs if os.path.basename(p).startswith(last_ep + '_')]
print(f'{len(pngs)} viz panels in {VIZ_DIR}; showing last epoch ({len(last)})')
n = min(6, len(last))
if n:
    fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(12, 3 * ((n + 1) // 2)))
    for ax, p in zip(np.ravel(axes), last[:n]):
        ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); ax.axis('off')
        ax.set_title(os.path.basename(p), fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('no viz (--viz_dir 껐거나 학습이 아직 안 돔)')

## 5. ONNX export + 검증

best 체크포인트를 `e2e.onnx`로. `--check`로 PyTorch vs onnxruntime 출력이 일치하는지 확인(maxΔ ~1e-5). 추론에서 `waypoints` 출력은 시각화 전용 — engine에선 무시.

In [ ]:
ONNX = '/content/e2e.onnx'
!{sys.executable} export_onnx.py --ckpt {CKPT} --out {ONNX} --check

## 6. 다운로드 → Jetson

`e2e_best.pt`(재학습/디버그용)와 `e2e.onnx`(배포용)를 받는다. **engine은 Jetson에서만** 빌드 (GPU 아키텍처가 박힌다):

```bash
# Jetson(Orin Nano)에서
/usr/src/tensorrt/bin/trtexec --onnx=e2e.onnx --fp16 --saveEngine=e2e.engine
```

이후 rover_lane 추론 노드가 `e2e.engine`을 쓴다 (PHASE2 7단계). 노드 전처리는 `dataset.composite_lane`/`composite_front`/`to_input_tensor`와 픽셀 단위로 동일해야 한다(ROI crop `LANE_CROP_TOP`, seg 색/alpha, bbox, BGR→RGB→ImageNet 순서).

In [ ]:
from google.colab import files
files.download(CKPT)
files.download(ONNX)